In [1]:
%load_ext autoreload 
%autoreload 2

import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(os.path.join(os.getcwd(), ".."))

import math
import numpy as np
import pandas as pd
from numpy import array
from numpy import array, arange, abs as np_abs
from numpy.fft import rfft, rfftfreq
from math import sin, pi
from scipy import signal
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import random

import importlib
import model as md
import view as vw

importlib.reload(md)
importlib.reload(vw)

from model.relevant_methods import phase_methods

c:\Users\Артем\Desktop\Вуз\Аспирантура\Диссертация\Алгоритм\Relaxation_frequency_phase_algorithm\notebooks\Phase_methods_analytics\Global_analytics\Sobol\..


In [22]:
from SALib.sample import saltelli
from SALib.analyze import sobol
import numpy as np
from tqdm import tqdm

def generate_Sobol(problem, phase_method, N=1024,seed=0):
    # 1. Генерация выборки Sobol
    
    np.random.seed(seed)  # фикс для Saltelli
    param_values = saltelli.sample(problem, N, calc_second_order=False)

    Y = []

    for i, p in enumerate(tqdm(param_values, desc=f"Sobol {phase_method.__name__}", unit=" шаг")):

        np.random.seed(i)
        phase_real, fs_mult, duration_T, bits, SNR = p #F, phase_real, fs_mult, duration_T, gen_unstable, bits = p

        F=400e3
        gen_unstable=0.1

        duration = duration_T / F
        fs = F * fs_mult

        t = np.linspace(0, duration, int(duration * fs))

        phase = md.generate_common_phase(
            t,
            F,
            gen_unstable,
            phase0_deg=0
        )

        U, I = md.generate_signals_from_phase(
            phase,
            phase_diff_deg=phase_real
        )

        #U = md.add_SNR(U, SNR)
        #I = md.add_SNR(I, SNR)

        U = md.adc_quantize(U, int(bits), 1)
        I = md.adc_quantize(I, int(bits), 1)

        phase_mes = phase_method(t, U, I, F)
        phase_error = abs(phase_mes) - abs(phase_real)

        Y.append(abs(phase_error))

    Y = np.array(Y)

    # 2. Анализ Sobol
    Si = sobol.analyze(problem, Y, calc_second_order=False)

    return Si

In [ ]:
# 1) Определяем параметры
problem = {
    'num_vars': 5,
    'names': ['φ', 'FsM','DT','Bit', 'SNR'], #: ['F', 'φ', 'FsM','DT','GU','Bit']
    'bounds': [
        #[300e3, 500e3],     # F
        [1,179],             # phase
        [3, 20],            # fs_mult
        [2, 200],           # duration_T
        #[0.01,10],             # generator instability
        [5,35],                  #bits
        [30,100]                  #SNR dB
    ]
}

In [24]:
def find_optimal_N(problem, phase_method, 
                   N_start=256, 
                   N_max=32768, 
                   step_factor=2,
                   tol=0.01):
    
    prev_S1 = None
    N = N_start

    history = []
    

    while N <= N_max:
        
        print(f"\n=== Testing N = {N} ===")
        
        Si = generate_Sobol(problem, phase_method, N=N,seed=42)
        S1 = Si['S1']
        
        history.append((N, S1))

        if prev_S1 is not None:
            diff = np.max(np.abs(S1 - prev_S1))
            print(f"Max ΔS1 = {diff:.5f}")

            if diff < tol:
                print(f"\n✅ Достаточный N найден: {N}")
                return N, history
        
        prev_S1 = S1
        N *= step_factor

    print("\n⚠️ Достигнут N_max без сходимости")
    return N, history

In [25]:
optimal_N, history = find_optimal_N(problem, md.get_phase_SWFR)

C:\Users\Артем\AppData\Local\Temp\ipykernel_2424\2340030385.py:10: DeprecationWarning: `salib.sample.saltelli` will be removed in SALib 1.5.1 Please use `salib.sample.sobol`
  param_values = saltelli.sample(problem, N, calc_second_order=False)



=== Testing N = 256 ===


Sobol get_phase_SWFR: 100%|██████████| 1792/1792 [00:00<00:00, 2631.73 шаг/s]



=== Testing N = 512 ===


Sobol get_phase_SWFR: 100%|██████████| 3584/3584 [00:01<00:00, 2619.90 шаг/s]


...

Max ΔS1 = 0.03798

=== Testing N = 16384 ===


Sobol get_phase_SWFR: 100%|██████████| 114688/114688 [00:44<00:00, 2578.08 шаг/s]


Max ΔS1 = 0.05476

=== Testing N = 32768 ===


Sobol get_phase_SWFR: 100%|██████████| 229376/229376 [01:30<00:00, 2538.24 шаг/s]


Max ΔS1 = 0.01491

⚠️ Достигнут N_max без сходимости
